### Anchor Assignments

Method Overview:
1. Preprocess each image using green channel + CLAHE + resize
2. Extract local features using SIFT
3. Match descriptors with Lowe's ratio test
4. Fit a homography with RANSAC
5. Score each anchor-test pair using:
   - number of inliers
   - inlier ratio
   - reprojection error
   - vessel-strucutre (Frangi) similarity
6. Assign each test image to the anchor with the best score



In [62]:
import sys
print(sys.executable)

import cv2
print(cv2.__version__)

/opt/miniconda3/envs/mia/bin/python
4.13.0


In [63]:
#!/usr/bin/env python3
"""
Requirements to install:
    pip install opencv-python numpy pandas
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd


VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

In [64]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Assign each test fundus image to an anchor image.")
    parser.add_argument("--anchors_dir", type=str, required=True, help="Directory containing anchor images")
    parser.add_argument("--tests_dir", type=str, required=True, help="Directory containing test images")
    parser.add_argument("--output_csv", type=str, required=True, help="Path to save grouping.csv")
    parser.add_argument(
        "--diagnostics_csv",
        type=str,
        default=None,
        help="Optional path to save detailed pairwise scores"
    )
    parser.add_argument(
        "--feature",
        type=str,
        default="sift",
        choices=["sift"],
        help="Feature extractor to use"
    )
    parser.add_argument(
        "--resize",
        type=int,
        default=512,
        help="Resize images to resize x resize before matching"
    )
    parser.add_argument(
        "--ratio_thresh",
        type=float,
        default=0.75,
        help="Lowe ratio test threshold"
    )
    parser.add_argument(
        "--ransac_thresh",
        type=float,
        default=5.0,
        help="RANSAC reprojection threshold in pixels"
    )
    parser.add_argument(
        "--clahe_clip",
        type=float,
        default=2.0,
        help="CLAHE clip limit"
    )
    parser.add_argument(
        "--nfeatures",
        type=int,
        default=2000,
        help="Number of features for SIFT or ORB"
    )
    return parser.parse_args()


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VALID_EXTENSIONS


def list_image_files(folder: Path) -> List[Path]:
    files = [p for p in sorted(folder.iterdir()) if is_image_file(p)]
    if not files:
        raise FileNotFoundError(f"No image files found in: {folder}")
    return files


def load_image(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Failed to read image: {path}")
    return img

In [65]:
def create_fundus_mask(gray: np.ndarray) -> np.ndarray:
    # Mask to suppress dark border/background.
    mask = (gray > 20).astype(np.uint8) * 255
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask


def preprocess_fundus(
    img_bgr: np.ndarray,
    resize_to: int = 512,
    clahe_clip: float = 2.0
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
        proc_img: preprocessed grayscale image
        mask: binary mask of valid fundus region
    """
    if img_bgr.ndim != 3 or img_bgr.shape[2] != 3:
        raise ValueError("Expected a color BGR image")

    # OpenCV loads BGR; green channel is index 1
    green = img_bgr[:, :, 1]

    # Resize first for consistency
    proc = cv2.resize(green, (resize_to, resize_to), interpolation=cv2.INTER_AREA)

    # Contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    proc = clahe.apply(proc)

    # Create mask from enhanced image
    mask = create_fundus_mask(proc)

    return proc, mask

def enhance_vessels(img):
    # Frangi-like effect using Sobel + CLAHE
    img = img.astype(np.float32) / 255.0
    sobelx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
    vessel = np.sqrt(sobelx**2 + sobely**2)
    return vessel


def make_detector(feature_type: str, nfeatures: int):
    if feature_type == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise RuntimeError(
                "SIFT is not available in your OpenCV build. "
                "Install a build that includes SIFT"
            )
        return cv2.SIFT_create(nfeatures=nfeatures)


    raise ValueError(f"Unsupported feature type: {feature_type}")


def extract_features(
    img: np.ndarray,
    mask: np.ndarray,
    detector
) -> Tuple[List[cv2.KeyPoint], Optional[np.ndarray]]:
    kp, des = detector.detectAndCompute(img, mask)
    return kp, des


def match_descriptors(
    des1: Optional[np.ndarray],
    des2: Optional[np.ndarray],
    feature_type: str,
    ratio_thresh: float
) -> List[cv2.DMatch]:
    if des1 is None or des2 is None:
        return []

    if len(des1) < 2 or len(des2) < 2:
        return []

    if feature_type == "sift":
        # FLANN for float descriptors
        index_params = dict(algorithm=1, trees=5)  # KD-tree
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)

        knn_matches = matcher.knnMatch(des1, des2, k=2)

    else:
        raise ValueError(f"Unsupported feature type: {feature_type}")

    good_matches: List[cv2.DMatch] = []
    for pair in knn_matches:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)

    return good_matches

In [66]:
def compute_reprojection_error(
    pts1: np.ndarray,
    pts2: np.ndarray,
    H: np.ndarray
) -> float:
    if len(pts1) == 0:
        return float("inf")

    pts1_proj = cv2.perspectiveTransform(pts1.reshape(-1, 1, 2), H).reshape(-1, 2)
    pts2 = pts2.reshape(-1, 2)
    err = np.sqrt(np.sum((pts1_proj - pts2) ** 2, axis=1))
    return float(np.mean(err)) if len(err) > 0 else float("inf")


def ransac_score(
    kp1: List[cv2.KeyPoint],
    kp2: List[cv2.KeyPoint],
    matches: List[cv2.DMatch],
    ransac_thresh: float
) -> Dict[str, float]:
    """
    kp1/matches query points are assumed to be from anchor image
    kp2/matches train points are assumed to be from test image
    """
    stats = {
        "num_matches": float(len(matches)),
        "num_inliers": 0.0,
        "inlier_ratio": 0.0,
        "mean_error": float("inf"),
    }

    if len(matches) < 4:
        return stats

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransac_thresh)

    if H is None or mask is None:
        return stats

    inlier_mask = mask.ravel().astype(bool)
    num_inliers = int(np.sum(inlier_mask))
    inlier_ratio = num_inliers / max(len(matches), 1)

    if num_inliers > 0:
        pts1_in = pts1[inlier_mask]
        pts2_in = pts2[inlier_mask]
        mean_error = compute_reprojection_error(pts1_in, pts2_in, H)
    else:
        mean_error = float("inf")

    stats["num_inliers"] = float(num_inliers)
    stats["inlier_ratio"] = float(inlier_ratio)
    stats["mean_error"] = float(mean_error)

    return stats

def vessel_similarity(img1, img2):
    v1 = enhance_vessels(img1).ravel()
    v2 = enhance_vessels(img2).ravel()
    if np.std(v1) < 1e-6 or np.std(v2) < 1e-6:
        return 0.0

    return float(np.corrcoef(v1, v2)[0, 1])


def final_pair_score(stats: Dict[str, float], vessel_sim= float) -> float:
    mean_error = stats["mean_error"]
    if np.isinf(mean_error) or np.isnan(mean_error):
        mean_error = 1e6

    score = (
        3.0 * stats["num_inliers"]
        + 40.0 * stats["inlier_ratio"]
        - 8.0 * mean_error
        + 40.0 * vessel_sim 
    )
    return float(score)


def build_feature_cache(
    image_paths: List[Path],
    detector,
    resize_to: int,
    clahe_clip: float
) -> Dict[str, Dict[str, object]]:
    cache: Dict[str, Dict[str, object]] = {}

    for path in image_paths:
        img_bgr = load_image(path)
        proc, mask = preprocess_fundus(img_bgr, resize_to=resize_to, clahe_clip=clahe_clip)
        kp, des = extract_features(proc, mask, detector)

        cache[path.name] = {
            "path": path,
            "proc": proc,
            "mask": mask,
            "kp": kp,
            "des": des,
        }

        print(f"Cached {path.name}: {len(kp)} keypoints")

    return cache


def assign_single_test_image(
    test_name: str,
    test_entry: Dict[str, object],
    anchor_cache: Dict[str, Dict[str, object]],
    feature_type: str,
    ratio_thresh: float,
    ransac_thresh: float
) -> Tuple[str, Dict[str, float], List[Dict[str, object]]]:
    pairwise_rows: List[Dict[str, object]] = []
    best_anchor_name: Optional[str] = None
    best_score = -float("inf")
    best_stats: Optional[Dict[str, float]] = None

    kp_test = test_entry["kp"]
    des_test = test_entry["des"]
    test_img = test_entry["proc"]

    for anchor_name, anchor_entry in anchor_cache.items():
        kp_anchor = anchor_entry["kp"]
        des_anchor = anchor_entry["des"]
        anchor_img = anchor_entry["proc"]

        matches = match_descriptors(
            des_anchor,
            des_test,
            feature_type=feature_type,
            ratio_thresh=ratio_thresh
        )

        stats = ransac_score(
            kp_anchor,
            kp_test,
            matches,
            ransac_thresh=ransac_thresh
        )

        vessel_sim = vessel_similarity(anchor_img, test_img)
        score = final_pair_score(stats, vessel_sim)

        row = {
            "test_image": test_name,
            "anchor_image": anchor_name,
            "score": score,
            "num_matches": int(stats["num_matches"]),
            "num_inliers": int(stats["num_inliers"]),
            "inlier_ratio": float(stats["inlier_ratio"]),
            "mean_error": float(stats["mean_error"]),
            "vessel_sim": float(vessel_sim),
        }
        pairwise_rows.append(row)

        if score > best_score:
            best_score = score
            best_anchor_name = anchor_name
            best_stats = stats

    if best_anchor_name is None or best_stats is None:
        raise RuntimeError(f"Failed to assign anchor for test image: {test_name}")

    return best_anchor_name, best_stats, pairwise_rows

In [67]:
def main() -> None:
    anchors_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/anchor_images")
    tests_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/test_images")
    output_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/grouping_results.csv")
    diagnostics_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/diagnostics_results.csv")

    feature = "sift"
    resize = 768
    ratio_thresh = 0.70
    ransac_thresh = 3.0
    clahe_clip = 2.0
    nfeatures = 4000

    if not anchors_dir.exists():
        raise FileNotFoundError(f"Anchors directory does not exist: {anchors_dir}")
    if not tests_dir.exists():
        raise FileNotFoundError(f"Tests directory does not exist: {tests_dir}")

    anchor_paths = list_image_files(anchors_dir)
    test_paths = list_image_files(tests_dir)

    print(f"Found {len(anchor_paths)} anchor images")
    print(f"Found {len(test_paths)} test images")

    detector = make_detector(feature, nfeatures)

    print("\nBuilding anchor cache...")
    anchor_cache = build_feature_cache(
        anchor_paths,
        detector=detector,
        resize_to=resize,
        clahe_clip=clahe_clip
    )

    print("\nBuilding test cache...")
    test_cache = build_feature_cache(
        test_paths,
        detector=detector,
        resize_to=resize,
        clahe_clip=clahe_clip
    )

    grouping_rows = []
    diagnostics_rows = []

    print("\nAssigning anchors...")
    for test_name, test_entry in test_cache.items():
        assigned_anchor, stats, pairwise_rows = assign_single_test_image(
            test_name=test_name,
            test_entry=test_entry,
            anchor_cache=anchor_cache,
            feature_type=feature,
            ratio_thresh=ratio_thresh,
            ransac_thresh=ransac_thresh
        )

        print(
            f"{test_name} -> {assigned_anchor} | "
            f"inliers={int(stats['num_inliers'])}, "
            f"inlier_ratio={stats['inlier_ratio']:.3f}, "
            f"mean_error={stats['mean_error']:.3f}"
        )

        grouping_rows.append({
            "test_image": test_name,
            "anchor_image": assigned_anchor,
        })
        diagnostics_rows.extend(pairwise_rows)

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    grouping_df = pd.DataFrame(grouping_rows)
    grouping_df.to_csv(output_csv, index=False)
    print(f"\nSaved grouping CSV to: {output_csv}")

    diagnostics_csv.parent.mkdir(parents=True, exist_ok=True)
    diagnostics_df = pd.DataFrame(diagnostics_rows)
    diagnostics_df.sort_values(["test_image", "score"], ascending=[True, False], inplace=True)
    diagnostics_df.to_csv(diagnostics_csv, index=False)
    print(f"Saved diagnostics CSV to: {diagnostics_csv}")


if __name__ == "__main__":
    main()

Found 5 anchor images
Found 25 test images

Building anchor cache...
Cached anchor_01.tiff: 2122 keypoints
Cached anchor_02.tiff: 664 keypoints
Cached anchor_03.tiff: 1229 keypoints
Cached anchor_04.tiff: 1112 keypoints
Cached anchor_05.tiff: 1084 keypoints

Building test cache...
Cached test_01.tiff: 206 keypoints
Cached test_02.tiff: 678 keypoints
Cached test_03.tiff: 3604 keypoints
Cached test_04.tiff: 1977 keypoints
Cached test_05.tiff: 756 keypoints
Cached test_06.tiff: 656 keypoints
Cached test_07.tiff: 1084 keypoints
Cached test_08.tiff: 3992 keypoints
Cached test_09.tiff: 680 keypoints
Cached test_10.tiff: 1706 keypoints
Cached test_11.tiff: 1245 keypoints
Cached test_12.tiff: 2217 keypoints
Cached test_13.tiff: 2017 keypoints
Cached test_14.tiff: 1225 keypoints
Cached test_15.tiff: 1157 keypoints
Cached test_16.tiff: 1111 keypoints
Cached test_17.tiff: 538 keypoints
Cached test_18.tiff: 3985 keypoints
Cached test_19.tiff: 1129 keypoints
Cached test_20.tiff: 575 keypoints
Cache